<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers

  Cloning https://github.com/KellerJordan/Muon to /tmp/pip-req-build-jsknuiw9
  Running command git clone --filter=blob:none --quiet https://github.com/KellerJordan/Muon /tmp/pip-req-build-jsknuiw9
  Resolved https://github.com/KellerJordan/Muon to commit 6399c658d3c4a3356ba823fa6664b10e23871068
  Preparing metadata (setup.py) ... done
  Using cached transformers-5.8.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.8.0-py3-none-any.whl (10.6 MB)
Using cached huggingface_hub-1.14.0-py3-none-any.whl (661 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Att

In [77]:
!hf auth login --force


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

In [79]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 342, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 342 (delta 10), reused 0 (delta 0), pack-reused 321 (from 2)
Receiving objects: 100% (342/342), 2.20 MiB | 7.10 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/Tiny-R2/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。**

In [89]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8  --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

✅ 配置同步完成！内存config已更新，config.py 文件已保存
✅ 配置同步完成！内存config已更新，config.py 文件已保存
No checkpoint found, starting from scratch
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: yingjun-xuda to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/Tiny-R2/Tiny-R2/wandb/run-20260507_185221-d86d6yat
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pretrain
wandb: ⭐️ View project at https://wandb.ai/yingjun-xuda/Tiny-R2-openweb
wandb: 🚀 View run at https://wandb.ai/yingjun-xuda/Tiny-R2-openweb/runs/d86d6yat
Loading HF dataset: karpathy/climbmix-400b-shuffle
Resolving data files: 100% 6543/6543 [00:00<00:00, 101971.68it/s]
IterableDataset({
    features: ['text'],
    num_shards: 6543
})
num Muon parameters: 951,454,128
num AdamW parameters: 

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

:直接使用Qwen3.5-9模型作为教师模式进行OPD训练，采用数据集mmlu_pro

In [ ]:

!python opd_train.py --batch_size 4 --ctx_len 2048 --hf_teacher_model Qwen/Qwen3.5-9B --student_ckpt "./opd_checkpoints/best_model_step_0.pt" --tokenizer_name Qwen/Qwen3.5-9B --dataset mmlu_pro

✅ 成功导入 Tiny-R2 核心模块 (model, config)
ℹ️  根据损失类型自动设置 opd_beta=0.0
ℹ️  验证BatchSize未指定，自动设置为: 8
📊 训练配置 | Micro-BS: 4 | 累积步数: 1 | 有效BS: 4
📚 选中数据集: mmlu_pro
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
✅ 进程组初始化成功（单卡模式，world_size=1）
📝 加载 Tokenizer
✅ 词汇量: 248077

📊 加载数据集: mmlu_pro
📊 数据集加载完成 | 样本数: 9626
📊 数据集加载完成 | 样本数: 2406

🤖 初始化学生模型
✅ 模型Norm层dtype对齐完成
num Muon parameters: 951,454,128
num AdamW parameters: 705,173,184

Model & Optimizer Summary

Total trainable parameters: 1656.627 M

--- Training Configuration ---
Effective BatchSize: 4
Micro BatchSize: 4
Gradient Accumulation Steps: 1
Loss Type: reverse_kl (beta=0.0)
Temperature: 1.0

--- Transformer Architecture ---
Number of layers: 6
Attention heads: 16
Embedding size: 1536
Layer 0: atten=Sparse, atten_mode=SWA, ffn=moe
Layer 1: atten=Sparse, atten_mode=SWA, ffn=moe
Layer 2: atten=Sparse, atten_mode=CSA, ffn=moe
Layer 3: atten=Sparse, atten_mode=HCA, ffn=moe
Layer 4: atten=Sparse, atten_mode=

In [ ]:
!python /content/Tiny-R2/sampler.py --checkpoint /content/Tiny-R2/opd_checkpoints/best_model_step_150.pt --prompt " 请回答以下多项选择题，只需给出正确选项 In the context of large language model (LLM) on-policy distillation (OPD), researchers aim to align a small student model with the full output distribution of a large, well-calibrated teacher model. The teacher model produces sharp, well-calibrated probabilities over the vocabulary, with meaningful probability mass assigned to both high-likelihood correct tokens and low-likelihood distractor tokens. The primary training objective is to ensure the student model fully replicates the teacher's entire predictive distribution, rather than only matching the most probable token. Which of the following loss functions is theoretically optimal for this objective, while also maintaining stable training dynamics for subquadratic student architectures?Options:A. Reverse Kullback-Leibler (KL) Divergence (KL(student || teacher))B. Forward Kullback-Leibler (KL) Divergence (KL(teacher || student)C. Jensen-Shannon (JS) Divergence D. Hard-label Negative Log-Likelihood (NLL) Loss E. Mean Squared Error (MSE) on raw logits"  --max_new_tokens 10 --temperature 0.1    --top_k 30  --top_p 0.95   --device cuda